In [17]:
# In this notebook:
#   load cleaned datasets
#   decide merge logic
#   create final analysis table
#   save integrated dataset
#   save:
#       final_dataset.csv

In [18]:
# ===========================
# 1. load cleaned datasets
# ===========================

import pandas as pd

gdp = pd.read_csv('../Data/Processed/gdp_clean.csv')
sales = pd.read_csv('../Data/Processed/sales_clean.csv')

# ensure correct datatypes
gdp['year'] = gdp['year'].astype(int)
sales['year'] = sales['year'].astype(int)

In [19]:
# ============================
# 2. Aggregate sales by year, country, and indicator
# ============================

sales_clean = sales.groupby(['country', 'year', 'indicator'], as_index=False)['sales_value'].mean()

In [20]:
# ===========================
# 3. Merge datasets
# ===========================

final_df = sales_clean.merge(
    gdp,
    on=["country", "year"],
    how="inner"
)

final_df = final_df.rename(columns={
    "unit": "gdp_unit"
})

# data cleaning, aggregates removal
aggregates = ["EA19", "EA20", "EA21", "EU27_2020"]
final_df = final_df[~final_df["country"].isin(aggregates)]

In [21]:
# =============================
# 4. Check the merged dataset
# ================================

print("Shape:", final_df.shape)
print("Columns:", final_df.columns.tolist())
print("Countries:", final_df["country"].nunique())
print("Indicators:", final_df["indicator"].unique())
print("Years:", final_df["year"].min(), "-", final_df["year"].max())

print(final_df.head())

Shape: (691, 6)
Columns: ['country', 'year', 'indicator', 'sales_value', 'gdp', 'gdp_unit']
Countries: 36
Indicators: ['G47' 'G476' 'G47_NF_HLTH']
Years: 2016 - 2025
  country  year indicator  sales_value      gdp    gdp_unit
0      AL  2016       G47    65.850000  10420.1  CLV10_MEUR
1      AL  2016      G476    67.433333  10420.1  CLV10_MEUR
2      AL  2017       G47    66.283333  10762.2  CLV10_MEUR
3      AL  2017      G476    87.400000  10762.2  CLV10_MEUR
4      AL  2018       G47    69.150000  11157.3  CLV10_MEUR


In [22]:
# ==============================
# 5. Verify aggregates are gone
# ==============================

print(sorted(final_df["country"].unique()))

['AL', 'AT', 'BA', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'ME', 'MK', 'MT', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'TR', 'UK']


In [23]:
# ==============================
# 6. save and export the data
# ==============================

final_df.to_csv('../Data/Processed/final_dataset.csv', index=False)
